In [1]:
import pandas as pd
import os
import glob
from sklearn.utils import shuffle

#### Configuration

In [14]:
FOLDERS = {
    'day1': '../../CSV-01-12/01-12/', # Folder 01-11/01-12
    'day2': '../../CSV-03-11/03-11/'  # Folder 03-11/03-12
}

N_TOTAL = 30000       # Target rows per class in the final master file
CHUNK_SIZE = 150000   # Number of rows to process at a time (saves RAM)
SEED = 42

# Features to drop immediately to prevent overfitting and save memory
DROP_COLS = ['Unnamed: 0', 'Flow ID', 'Source IP', 'Source Port', 'Destination IP',
              'Destination Port', 'Timestamp', 'SimillarHTTP', 'Fwd Header Length.1']


In [15]:
label_map = {
    # BENIGN
    'BENIGN': 'BENIGN',

    # DrDoS => normalized
    'DrDoS_DNS': 'DNS',
    'DrDoS_LDAP': 'LDAP',
    'DrDoS_MSSQL': 'MSSQL',
    'DrDoS_NTP': 'NTP',
    'DrDoS_NetBIOS': 'NetBIOS',
    'DrDoS_SNMP': 'SNMP',
    'DrDoS_SSDP': 'SSDP',
    'DrDoS_UDP': 'UDP',

    # other label
    'DNS': 'DNS',
    'LDAP': 'LDAP',
    'MSSQL': 'MSSQL',
    'NetBIOS': 'NetBIOS',
    'NTP': 'NTP',
    'SNMP': 'SNMP',
    'SSDP': 'SSDP',
    'UDP': 'UDP',

    'UDP-lag': 'UDPLag',
    'UDPLag': 'UDPLag',
    'Syn': 'Syn',
    'TFTP': 'TFTP',
    'WebDDoS': 'WebDDoS',
    'Portmap': 'Portmap'
}

#### Chunk-baase random sampler (Helper)

In [16]:
def get_random_sample_from_file(filepath, target_label, n_needed):
    """Reads file, drops specific columns, normalizes labels, and samples."""
    print("..Extracting dataset from ", filepath)
    temp_list = []
    
    # Usecols filter to save memory during read
    try:
        # 1. Get headers to filter columns (handling potential leading spaces)
        raw_cols = pd.read_csv(filepath, nrows=0).columns
        use_cols = [c for c in raw_cols if c.strip() not in DROP_COLS]
        
        
        for chunk in pd.read_csv(filepath, chunksize=CHUNK_SIZE, usecols=use_cols, low_memory=False):
            chunk.columns = chunk.columns.str.strip() # Clean header whitespace
            
            # 2. Normalize labels using the map
            # We use .get() to keep the original label if it's not in our map (safeguard)
            chunk['Label'] = chunk['Label'].map(lambda x: label_map.get(x.strip() if isinstance(x, str) else x, x))

            # 3. Filter for the target (normalized) label
            matches = chunk[chunk['Label'] == target_label]

            
            
            if not matches.empty:
                # Sample from this chunk (oversample slightly to ensure we have enough for the final pool)
                sample_n = min(len(matches), n_needed)
                temp_list.append(matches.sample(n=sample_n, random_state=SEED))
        
        if not temp_list:
            return pd.DataFrame()
            
        # Combine all chunk-samples and take the final random target
        combined = pd.concat(temp_list, ignore_index=True)
        return combined.sample(n=min(len(combined), n_needed), random_state=SEED)
    
    except Exception as e:
        print(f"Error processing {filepath}: {e}")
        return pd.DataFrame()


In [17]:
# ── 3. CLASS MAPPING ────────────────────────────────────────────────────────
# Based on your provided structure
shared_classes = ['LDAP', 'MSSQL', 'NetBIOS', 'SYN', 'UDP', 'UDPLag']
day1_only = ['DNS', 'NTP', 'SNMP', 'SSDP', 'TFTP', 'WebDDoS']
day2_only = ['Portmap']

#### EXTRACTION PIPELINE

In [18]:
final_pool = []

- shared classes

In [19]:
for label in shared_classes:
    print(f"Extracting {label}")
    # Note: Using wildcards to find files (e.g., *LDAP*.csv)
    f1_list = glob.glob(os.path.join(FOLDERS['day1'], f"*{label}*"))
    f2_list = glob.glob(os.path.join(FOLDERS['day2'], f"*{label}*"))
    
    if f1_list and f2_list:
        s1 = get_random_sample_from_file(f1_list[0], label, N_TOTAL // 2)
        s2 = get_random_sample_from_file(f2_list[0], label, N_TOTAL // 2)
        final_pool.append(pd.concat([s1, s2], ignore_index=True))


Extracting LDAP
..Extracting dataset from  ../../CSV-01-12/01-12\DrDoS_LDAP.csv
..Extracting dataset from  ../../CSV-03-11/03-11\LDAP.csv
Extracting MSSQL
..Extracting dataset from  ../../CSV-01-12/01-12\DrDoS_MSSQL.csv
..Extracting dataset from  ../../CSV-03-11/03-11\MSSQL.csv
Extracting NetBIOS
..Extracting dataset from  ../../CSV-01-12/01-12\DrDoS_NetBIOS.csv
..Extracting dataset from  ../../CSV-03-11/03-11\NetBIOS.csv
Extracting SYN
..Extracting dataset from  ../../CSV-01-12/01-12\Syn.csv
..Extracting dataset from  ../../CSV-03-11/03-11\Syn.csv
Extracting UDP
..Extracting dataset from  ../../CSV-01-12/01-12\DrDoS_UDP.csv
..Extracting dataset from  ../../CSV-03-11/03-11\UDP.csv
Extracting UDPLag
..Extracting dataset from  ../../CSV-01-12/01-12\UDPLag.csv
..Extracting dataset from  ../../CSV-03-11/03-11\UDPLag.csv


- unique classes

In [20]:
for label in day1_only:
    print(f"Extracting {label}")
    f_list = glob.glob(os.path.join(FOLDERS['day1'], f"*{label}*"))
    if f_list:
        final_pool.append(get_random_sample_from_file(f_list[0], label, N_TOTAL))

for label in day2_only:
    print(f"Extracting {label}")
    f_list = glob.glob(os.path.join(FOLDERS['day2'], f"*{label}*"))
    if f_list:
        final_pool.append(get_random_sample_from_file(f_list[0], label, N_TOTAL))


Extracting DNS
..Extracting dataset from  ../../CSV-01-12/01-12\DrDoS_DNS.csv
Extracting NTP
..Extracting dataset from  ../../CSV-01-12/01-12\DrDoS_NTP.csv
Extracting SNMP
..Extracting dataset from  ../../CSV-01-12/01-12\DrDoS_SNMP.csv
Extracting SSDP
..Extracting dataset from  ../../CSV-01-12/01-12\DrDoS_SSDP.csv
Extracting TFTP
..Extracting dataset from  ../../CSV-01-12/01-12\TFTP.csv
Extracting WebDDoS
Extracting Portmap
..Extracting dataset from  ../../CSV-03-11/03-11\Portmap.csv


- consolidate benign

In [21]:
print("Extracting BENIGN (Global Pool)")
benign_samples = []
all_files = glob.glob(os.path.join(FOLDERS['day1'], "*.csv")) + glob.glob(os.path.join(FOLDERS['day2'], "*.csv"))

BEGIN_N_TOTAL = 60000
for f in all_files:
    # Take a small portion of benign from every file for maximum variety
    benign_samples.append(get_random_sample_from_file(f, 'BENIGN', BEGIN_N_TOTAL // len(all_files)))

master_benign = pd.concat(benign_samples, ignore_index=True).drop_duplicates()
final_pool.append(master_benign.sample(n=min(len(master_benign), N_TOTAL), random_state=SEED))


Extracting BENIGN (Global Pool)
..Extracting dataset from  ../../CSV-01-12/01-12\DrDoS_DNS.csv
..Extracting dataset from  ../../CSV-01-12/01-12\DrDoS_LDAP.csv
..Extracting dataset from  ../../CSV-01-12/01-12\DrDoS_MSSQL.csv
..Extracting dataset from  ../../CSV-01-12/01-12\DrDoS_NetBIOS.csv
..Extracting dataset from  ../../CSV-01-12/01-12\DrDoS_NTP.csv
..Extracting dataset from  ../../CSV-01-12/01-12\DrDoS_SNMP.csv
..Extracting dataset from  ../../CSV-01-12/01-12\DrDoS_SSDP.csv
..Extracting dataset from  ../../CSV-01-12/01-12\DrDoS_UDP.csv
..Extracting dataset from  ../../CSV-01-12/01-12\Syn.csv
..Extracting dataset from  ../../CSV-01-12/01-12\TFTP.csv
..Extracting dataset from  ../../CSV-01-12/01-12\UDPLag.csv
..Extracting dataset from  ../../CSV-03-11/03-11\LDAP.csv
..Extracting dataset from  ../../CSV-03-11/03-11\MSSQL.csv
..Extracting dataset from  ../../CSV-03-11/03-11\NetBIOS.csv
..Extracting dataset from  ../../CSV-03-11/03-11\Portmap.csv
..Extracting dataset from  ../../CSV-03-1

#### Final Merg and Save

In [ ]:
OUTPUT_FILE = 'CICDDOS_Final_Dataset.csv'


In [22]:
OUTPUT_FILE = 'CICDDOS_Final_Dataset.csv'

master_df = pd.concat(final_pool, ignore_index=True)
master_df = shuffle(master_df, random_state=SEED).reset_index(drop=True)

master_df.to_csv("../dataset/CICDDOS_Final_Dataset.csv", index=False)
print(f"\n[SUCCESS] Master Dataset Saved: {OUTPUT_FILE}")
print(f"Total rows: {len(master_df)}")
print("\nFinal Class Distribution:")
print(master_df['Label'].value_counts())



[SUCCESS] Master Dataset Saved: CICDDOS_Final_Dataset.csv
Total rows: 346873

Final Class Distribution:
Label
MSSQL      30000
NetBIOS    30000
SNMP       30000
BENIGN     30000
NTP        30000
UDP        30000
Portmap    30000
DNS        30000
LDAP       30000
SSDP       30000
TFTP       30000
UDPLag     16873
Name: count, dtype: int64
